# Create Institute for Evaluation of Labour Market and Education Policy Awards

**RE-SOURCED 2026-07-12 from SweCRIS** (previously a scrape of the ~16 grants listed on ifau.se's awarded-research-grants page). SweCRIS carries IFAU's full grant history: **131 projects 2005-2026 with 100% SEK amounts**, dates, abstracts and partial PI/ORCID data.

provenance `ifau`, priority 338 (both unchanged — the DELETE below cleanly replaces the old ifau.se rows). F4320327653 (Path A).

**Prerequisites:** run `scripts/local/ifau_to_s3.py`.

**Data source:** https://swecris-api.vr.se (SweCRIS API, org nr 202100-4946)
**S3 location:** `s3a://openalex-ingest/awards/ifau/ifau_grants.parquet` (unchanged)

**Funder:** funder_id 4320327653, display_name "Institutet för arbetsmarknads- och utbildningspolitisk utvärdering", ROR https://ror.org/015zanq20

**Mapping notes:**
- `funder_award_id` = SweCRIS projectId with the `_IFAU` suffix stripped (e.g. `108/2010`). NOTE: ids change vs the old ifau.se slug ids — award ids are re-minted, which is expected on re-resolution.
- `amount` = `fundingsSek`, **SEK**; zeros treated as not-published.
- PI present on ~22% of rows (SweCRIS coverage; the old 16-row scrape had 100% PI but 8x fewer grants).


## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.ifau_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/ifau/ifau_grants.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.ifau_raw;

In [ ]:
%sql
-- Step 1.5: inspect raw data before transforming
DESCRIBE openalex.awards.ifau_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.ifau_raw LIMIT 5;

In [ ]:
%sql
-- Step 1.6 funder existence check (Path A: F4320* must return exactly 1 row)
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320327653;

## Step 2: Create Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.ifau_awards
USING delta
AS
WITH
the_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320327653
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(REGEXP_REPLACE(TRIM(g.project_id), '_[A-Za-z]+$', ''))))) % 9000000000 as id,
        COALESCE(NULLIF(TRIM(g.title_english), ''), NULLIF(TRIM(g.title), '')) as display_name,
        COALESCE(NULLIF(TRIM(g.abstract_english), ''), NULLIF(TRIM(g.abstract), '')) as description,
        f.funder_id,
        REGEXP_REPLACE(TRIM(g.project_id), '_[A-Za-z]+$', '') as funder_award_id,
        NULLIF(TRY_CAST(g.amount AS DOUBLE), 0) as amount,
        'SEK' as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,
        CASE
            WHEN LOWER(COALESCE(g.type_of_award, '')) LIKE '%fellow%' THEN 'fellowship'
            WHEN LOWER(COALESCE(g.type_of_award, '')) LIKE '%stipend%' THEN 'fellowship'
            WHEN LOWER(COALESCE(g.type_of_award, '')) LIKE '%position%' THEN 'fellowship'
            WHEN LOWER(COALESCE(g.type_of_award, '')) LIKE '%infrastructure%' THEN 'infrastructure'
            WHEN LOWER(COALESCE(g.type_of_award, '')) LIKE '%project%' THEN 'research'
            ELSE 'grant'
        END as funding_type,
        NULLIF(TRIM(g.type_of_award), '') as funder_scheme,
        'ifau' as provenance,
        TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
        TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
        YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')) as start_year,
        YEAR(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd')) as end_year,
        CASE
            WHEN g.pi_family_name IS NOT NULL AND TRIM(g.pi_family_name) != '' THEN
                struct(
                    NULLIF(TRIM(g.pi_given_name), '') as given_name,
                    TRIM(g.pi_family_name) as family_name,
                    NULLIF(TRIM(g.pi_orcid), '') as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        NULLIF(TRIM(g.coordinating_organisation), '') as name,
                        'Sweden' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
        CONCAT('https://www.vr.se/swecris#/project/', TRIM(g.project_id)) as landing_page_url,
        CAST(NULL AS STRING) as doi,
        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(REGEXP_REPLACE(TRIM(g.project_id), '_[A-Za-z]+$', ''))))) % 9000000000) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.ifau_raw g
    CROSS JOIN the_funder f
    WHERE g.project_id IS NOT NULL AND TRIM(g.project_id) != ''
)
SELECT * FROM awards_transformed;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data
-- (this DELETE also clears the pre-2026-07 first-party rows: same provenance+priority)
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'ifau' AND priority = 338;

-- Insert into openalex_awards_raw with priority
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    338 as priority  -- IFAU priority (unchanged)
FROM openalex.awards.ifau_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) as total_awards FROM openalex.awards.ifau_awards;

In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_abstract,
    COUNT(amount) as has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) as pct_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator) as has_pi,
    MIN(amount) as min_amount,
    ROUND(AVG(amount), 0) as avg_amount,
    MAX(amount) as max_amount,
    ROUND(SUM(amount)/1e6, 1) as total_amount_msek
FROM openalex.awards.ifau_awards;

In [ ]:
%sql
-- 6.4a PI frequency check
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.ifau_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.ifau_awards
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC LIMIT 25;

In [ ]:
%sql
SELECT COUNT(*) as in_shared_raw
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'ifau' AND priority = 338;